# 03 · Train S1-PII
Trains one (variant, seed) per run on an A100 or L4 in bf16. Checkpoints go to local disk and are mirrored to Drive; **re-running the train cell after a disconnect resumes exactly** (model, optimizer, scheduler, RNG, sampler position). Final weights and a manifest (data counts, code SHA, weights hash) land in `MyDrive/s1pii/models/<variant>-s<seed>/final`.

Prereg: run `all-sources` and `no-nemotron`, seeds 1 to 3. Tag `prereg-v0` must exist before the first real run.

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')

def git(*args):
    # The token (Colab secret GH_TOKEN: fine-grained, read-only on anitksahu/s1-pii) is passed
    # through GIT_ASKPASS, never on the command line, and errors are printed redacted.
    try:
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    if not tok:   # public repo: no token needed
        r = subprocess.run(['git', *args], capture_output=True, text=True)
        if r.returncode: raise RuntimeError('git ' + args[0] + ' failed: ' + r.stderr[-800:])
        return r.stdout
    askpass = pathlib.Path('/tmp/askpass.sh'); askpass.write_text('#!/bin/sh\necho "$S1PII_GIT_TOKEN"\n'); askpass.chmod(0o700)
    env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'S1PII_GIT_TOKEN': tok, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('git ' + args[0] + ' failed: ' + (r.stderr or '').replace(tok, '***')[-800:])
    return r.stdout

if not REPO.exists():
    git('clone', 'https://x-access-token@github.com/anitksahu/s1-pii.git', str(REPO))
else:
    git('-C', str(REPO), 'pull', '--ff-only')
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data,model]' && python -c "import subprocess;t=subprocess.run(['git','tag','-l','prereg-v0'],capture_output=True,text=True).stdout.strip();assert t=='prereg-v0','tag prereg-v0 before training'"

In [ ]:
VARIANT, SEED = 'all-sources', 1      # then no-nemotron; seeds 1..3
MAX_STEPS = None                         # set e.g. 50 for a smoke run
import os
OUT = f'/content/ckpt/{VARIANT}-s{SEED}'
MIRROR = f'/content/drive/MyDrive/s1pii/ckpt/{VARIANT}-s{SEED}'
args = f'--variant {VARIANT} --seed {SEED} --out {OUT} --mirror {MIRROR}' + (f' --max-steps {MAX_STEPS}' if MAX_STEPS else '')
!python -m s1pii.model.train {args}

In [ ]:
!rsync -a {OUT}/final/ /content/drive/MyDrive/s1pii/models/{VARIANT}-s{SEED}/final/ && cat /content/drive/MyDrive/s1pii/models/{VARIANT}-s{SEED}/final/s1_manifest.json | head -40